# PneumoScan — Détection de pneumonie (RSNA) avec YOLO — v3

Pipeline : DICOM → PNG → format YOLO → entraînement → évaluation (P, R, mAP@50, mAP@50-95).

**Changements par rapport à la v2 (mAP@0.5 = 0,396, F1 = 47,8% au seuil optimal) :**
1. `RATIO_NEGATIFS` passe de 1.0 à **2.0** — plus proche du vrai ratio clinique du dataset RSNA (~3:1), donne plus d'exemples négatifs au modèle
2. `epochs` passe de 45 à **100**, `patience` de 15 à **25** — laisse plus de temps au modèle pour converger
3. Le reste (modèle, hyperparamètres d'augmentation) reste identique à la v2, pour isoler l'effet de ces deux changements

⚠️ Active le GPU : *Settings → Accelerator → GPU T4 x2 (ou P100)* — vérifie-le à CHAQUE nouvelle version avant de lancer un commit, Kaggle ne le reporte pas toujours automatiquement.

⚠️ Durée totale estimée : **5 à 7 heures** (plus de données négatives + plus d'epochs). Utilise « Save Version → Save & Run All (Commit) », pas une exécution interactive — ça tourne en arrière-plan et résiste aux déconnexions.

In [ ]:
!pip install -q ultralytics pydicom

import torch
print("GPU disponible :", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ Active le GPU dans Settings")


## 1. Chargement des annotations

In [ ]:
import os, random, shutil
import numpy as np
import pandas as pd

RSNA = '/kaggle/input/competitions/rsna-pneumonia-detection-challenge'
WORK = '/kaggle/working/dataset'

labels = pd.read_csv(f'{RSNA}/stage_2_train_labels.csv')
print("Lignes :", len(labels))
print("Patients uniques :", labels['patientId'].nunique())
print(labels.groupby('patientId')['Target'].max().value_counts())
labels.head()


## 2. Sélection d'un dataset équilibré (ratio ajusté)

**v3** : `RATIO_NEGATIFS = 2.0` au lieu de 1.0 — deux négatifs pour un positif, plus proche du déséquilibre réel du dataset RSNA (~20% de cas positifs), tout en restant moins extrême que le ratio brut (~1:3.3) pour ne pas retomber dans le piège "le modèle apprend à ne rien prédire".

In [ ]:
RATIO_NEGATIFS = 2.0   # v3 : etait 1.0. Piste suivante si besoin : 3.0 (ratio reel du dataset)
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# boites regroupees par patient
boites = {}
for r in labels.itertuples(index=False):
    d = boites.setdefault(r.patientId, [])
    if r.Target == 1 and not pd.isna(r.x):
        d.append([r.x, r.y, r.width, r.height])

positifs = [p for p, b in boites.items() if len(b) > 0]
negatifs = [p for p, b in boites.items() if len(b) == 0]
negatifs = random.sample(negatifs, min(len(negatifs), int(len(positifs) * RATIO_NEGATIFS)))

ids = positifs + negatifs
cible = [1] * len(positifs) + [0] * len(negatifs)
print(f"Positifs : {len(positifs)} | Negatifs retenus : {len(negatifs)} | Total : {len(ids)}")


In [ ]:
from sklearn.model_selection import train_test_split

# split stratifie par patient : aucune fuite de donnees entre train et val
train_ids, val_ids = train_test_split(ids, test_size=0.2, random_state=SEED, stratify=cible)
print("Train :", len(train_ids), "| Val :", len(val_ids))


## 3. Conversion DICOM → PNG + labels YOLO

Format YOLO detection : `classe cx cy w h` normalises entre 0 et 1. Un fichier `.txt` vide = image negative (arriere-plan), ce qui est utile et attendu par Ultralytics.

⚠️ Avec le ratio 2.0, il y a plus d'images a convertir qu'en v2 — cette etape va prendre un peu plus longtemps.

In [ ]:
import pydicom
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

TAILLE_PNG = 640   # on redimensionne des la conversion : disque + entrainement plus rapides

for split in ['train', 'val']:
    os.makedirs(f'{WORK}/images/{split}', exist_ok=True)
    os.makedirs(f'{WORK}/labels/{split}', exist_ok=True)

def convertir(pid, split):
    ds = pydicom.dcmread(f'{RSNA}/stage_2_train_images/{pid}.dcm')
    img = ds.pixel_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-6) * 255.0
    h, w = img.shape
    im = Image.fromarray(img.astype(np.uint8))
    if TAILLE_PNG:
        im = im.resize((TAILLE_PNG, TAILLE_PNG), Image.BILINEAR)
    im.save(f'{WORK}/images/{split}/{pid}.png')

    lignes = []
    for x, y, bw, bh in boites[pid]:
        cx, cy = (x + bw / 2) / w, (y + bh / 2) / h
        nw, nh = bw / w, bh / h
        lignes.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
    with open(f'{WORK}/labels/{split}/{pid}.txt', 'w') as f:
        f.write('\n'.join(lignes))

for split, liste in [('train', train_ids), ('val', val_ids)]:
    with ThreadPoolExecutor(max_workers=8) as ex:
        list(tqdm(ex.map(lambda p: convertir(p, split), liste), total=len(liste), desc=split))

print("Images train :", len(os.listdir(f'{WORK}/images/train')))
print("Images val   :", len(os.listdir(f'{WORK}/images/val')))


In [ ]:
data_yaml = f"""path: {WORK}
train: images/train
val: images/val
nc: 1
names:
  0: pneumonie
"""
with open(f'{WORK}/data.yaml', 'w') as f:
    f.write(data_yaml)
print(data_yaml)


## 4. Vérification visuelle des annotations (à mettre dans la présentation)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

exemples = [p for p in train_ids if len(boites[p]) > 0][:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, pid in zip(axes.ravel(), exemples):
    im = Image.open(f'{WORK}/images/train/{pid}.png')
    ax.imshow(im, cmap='gray')
    for ligne in open(f'{WORK}/labels/train/{pid}.txt').read().splitlines():
        _, cx, cy, nw, nh = map(float, ligne.split())
        W, H = im.size
        ax.add_patch(patches.Rectangle(((cx - nw/2)*W, (cy - nh/2)*H), nw*W, nh*H,
                                       fill=False, edgecolor='red', linewidth=2))
    ax.set_title(pid[:12], fontsize=9); ax.axis('off')
plt.suptitle("Verification des annotations converties au format YOLO")
plt.tight_layout(); plt.show()


## 5. Entraînement (v3)

**Changements** : `epochs=100` (au lieu de 45), `patience=25` (au lieu de 15) — le reste des hyperparametres reste identique a la v2, pour isoler proprement l'effet du ratio de negatifs et de la duree d'entrainement.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')      # alternative si le temps le permet : 'yolo11l.pt' (plus grand, plus lent)

resultats = model.train(
    data=f'{WORK}/data.yaml',
    epochs=100,                  # v3 : etait 45
    imgsz=640,
    batch=16,
    project='/kaggle/working/pneumoscan_runs',
    name='pneumoscan_yolo11m_v3',
    exist_ok=True,
    seed=SEED,

    # optimisation
    optimizer='AdamW',
    lr0=0.001, lrf=0.01, cos_lr=True,
    warmup_epochs=3.0, weight_decay=0.0005,
    patience=25,                 # v3 : etait 15

    # ponderation des pertes (localisation privilegiee)
    box=7.5, cls=0.5, dfl=1.5,

    # augmentation adaptee aux radios
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.3,
    degrees=7.0, translate=0.1, scale=0.25,
    shear=0.0, perspective=0.0,
    fliplr=0.5, flipud=0.0,
    mosaic=0.6, close_mosaic=10, mixup=0.0, erasing=0.2,

    single_cls=True, amp=True, plots=True, workers=4, cache=False,
)


## 6. Évaluation — les chiffres à présenter

In [ ]:
RUN = '/kaggle/working/pneumoscan_runs/pneumoscan_yolo11m_v3'
best = YOLO(f'{RUN}/weights/best.pt')

m = best.val(data=f'{WORK}/data.yaml', imgsz=640, split='val', plots=True)

tableau = pd.DataFrame([{
    'Precision (P)': round(m.box.mp, 4),
    'Rappel (R)':    round(m.box.mr, 4),
    'mAP@0.5':       round(m.box.map50, 4),
    'mAP@0.5:0.95':  round(m.box.map, 4),
    'F1':            round(2 * m.box.mp * m.box.mr / (m.box.mp + m.box.mr + 1e-9), 4),
}])
display(tableau)
tableau.to_csv('/kaggle/working/metriques_finales_v3.csv', index=False)


In [ ]:
# Balayage du seuil de confiance : sert a justifier le seuil retenu dans la presentation
lignes = []
for conf in [0.10, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]:
    r = best.val(data=f'{WORK}/data.yaml', imgsz=640, conf=conf, plots=False, verbose=False)
    p, rec = r.box.mp, r.box.mr
    lignes.append({'conf': conf, 'Precision': round(p, 4), 'Rappel': round(rec, 4),
                   'F1': round(2*p*rec/(p+rec+1e-9), 4), 'mAP@0.5': round(r.box.map50, 4)})

sweep = pd.DataFrame(lignes)
display(sweep)
meilleur = sweep.loc[sweep['F1'].idxmax()]
print(f"\nMeilleur compromis : conf={meilleur['conf']} -> P={meilleur['Precision']}, R={meilleur['Rappel']}, F1={meilleur['F1']}")
sweep.to_csv('/kaggle/working/balayage_seuil_v3.csv', index=False)


In [ ]:
# Graphiques generes par Ultralytics : a inserer directement dans les diapositives
from IPython.display import Image as IPImage, display as dsp
for f in ['results.png', 'PR_curve.png', 'P_curve.png', 'R_curve.png',
          'F1_curve.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    chemin = f'{RUN}/{f}'
    if os.path.exists(chemin):
        print('—', f); dsp(IPImage(filename=chemin, width=780))


In [ ]:
# Predictions vs verite terrain sur quelques cas de validation
cas = [p for p in val_ids if len(boites[p]) > 0][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, pid in zip(axes.ravel(), cas):
    chemin = f'{WORK}/images/val/{pid}.png'
    im = Image.open(chemin); W, H = im.size
    ax.imshow(im, cmap='gray')
    for ligne in open(f'{WORK}/labels/val/{pid}.txt').read().splitlines():   # verite (vert)
        _, cx, cy, nw, nh = map(float, ligne.split())
        ax.add_patch(patches.Rectangle(((cx-nw/2)*W, (cy-nh/2)*H), nw*W, nh*H,
                                       fill=False, edgecolor='lime', linewidth=2))
    pred = best.predict(chemin, conf=float(meilleur['conf']), verbose=False)[0]
    for b in pred.boxes:                                                     # prediction (rouge)
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                       edgecolor='red', linewidth=2, linestyle='--'))
        ax.text(x1, y1-5, f"{float(b.conf):.2f}", color='red', fontsize=9)
    ax.axis('off')
plt.suptitle("Vert = verite terrain | Rouge pointille = prediction du modele")
plt.tight_layout(); plt.savefig('/kaggle/working/comparaison_predictions_v3.png', dpi=130); plt.show()


## 7. Comparaison v2 vs v3

Cellule facultative : si tu as encore le CSV `metriques_finales.csv` de la v2 disponible dans ce notebook (par exemple en le rattachant comme Input, ou en le collant a la main), tu peux comparer directement les deux versions.

In [ ]:
# Comparaison manuelle - remplace les valeurs par celles de ta v2 si tu veux les avoir cote a cote
v2 = {'Precision (P)': 0.4321, 'Rappel (R)': 0.5345, 'mAP@0.5': 0.3961, 'F1': 0.4778}
v3 = tableau.iloc[0].to_dict()

comparaison = pd.DataFrame([v2, v3], index=['v2 (ratio 1:1, 45 epochs)', 'v3 (ratio 1:2, 100 epochs)'])
display(comparaison)


In [ ]:
from IPython.display import FileLink
display(FileLink(f'{RUN}/weights/best.pt'))
display(FileLink('/kaggle/working/metriques_finales_v3.csv'))
display(FileLink('/kaggle/working/balayage_seuil_v3.csv'))
